#### Objectif

Ce notebook sert de workflow d'orchestration pour les statistiques par condition à partir des exports Morlet déjà calculés. Il tourne indépendamment du pipeline TRC/Morlet, à condition que les dossiers de results_morlet_exploratoire existent déjà.

Principe général :

- on recharge, pour chaque session, les cartes TF par canal déjà exportées ;
- on groupe les essais selon group_label (cog+, controle, negatif) et/ou selon les sous-catégories de cog_labels ;
- on teste, pour chaque bin temps-fréquence, si la métrique normalisée à la baseline diffère de 0 à travers les essais d'une condition ;
- on sauvegarde les cartes moyennes par condition, les masques significatifs et les figures.

Pour l'instant, l'organisation est de type : 
results_stats_conditions/
    SESSION_1/
        condition_main/
            logratio/
                cog+/
                    local/
                        mean_map.npy
                        median_map.npy
                        observations_table.csv
                        ...
                    distant/
                        mean_map.npy
                        median_map.npy
                        observations_table.csv
                        ...
                controle/
                    local/
                    distant/
                negatif/
                    local/
                    distant/

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lfp_utils import (
    StatsConfig,
    load_session_exports,
    build_main_condition_index,
    build_cog_subcategory_index,
    stack_condition_locality_maps,
    stack_condition_locality_maps_across_sessions,
    run_session_condition_stats,
    run_pooled_condition_stats,
    run_all_sessions_stats,
    wilcoxon_map_against_zero,
    fdr_bh,
    cluster_1samp_map_against_zero,
    generate_pooled_band_lineplots
)

In [2]:
# configuration
stats_cfg = StatsConfig(
    input_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_morlet_exploratoire",
    output_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_stats_conditions",

    metric="logratio",

    run_wilcoxon_fdr=True,
    run_cluster_perm=True,

    alpha_fdr=0.05,
    cluster_alpha=0.05,
    n_permutations=2000,
    cluster_threshold_p=0.05,
    tail=0,
    seed=13,

    min_trials_per_condition=5,
    make_main_groups=True,
    make_cog_subgroups=True,
    keep_main_groups_in_subgroup_mode=True,
    localities_to_test=("local", "distant"),

    fallback_compute_metric_from_raw=True,
    baseline_stat_fallback="median",
    eps=1e-12,

    group_across_sessions=True,
    also_run_per_session=False,
    pooled_output_subdir="pooled_across_sessions",

    save_figures=True,
    figure_dpi=150,
    cmap="RdBu_r",
    add_cluster_contour=True,

    make_band_lineplots=True,

    verbose=True,
)

In [3]:
# quelles sessions dispo en Morlet ?
input_root = Path(stats_cfg.input_root)
session_dirs = sorted([p for p in input_root.iterdir() if p.is_dir()])
print(f"{len(session_dirs)} sessions Morlet trouvées")
[p.name for p in session_dirs]

2 sessions Morlet trouvées


['P101_DC54_stim2', 'P64_BR34_stim2']

In [4]:
# pour avoir seulement les lineplots récap par bande de fréquence
generate_pooled_band_lineplots(stats_cfg)

#### Stats sur une session

In [7]:
session_dir = session_dirs[0]
session_dir

PosixPath('/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_morlet_exploratoire/P101_DC54_stim2')

In [8]:
exports = load_session_exports(session_dir)

print("Session :", exports["session"])
print("freqs shape :", exports["freqs"].shape)
print("post_times shape :", exports["post_times"].shape)
print("n_stims :", len(exports["stims_df"]))
print("n_bipolar_channels :", len(exports["metadata"]["bipolar_names"]))

exports["stims_df"]

Session : P101_DC54_stim2
freqs shape : (20,)
post_times shape : (384,)
n_stims : 42
n_bipolar_channels : 89


,label_stim,t_start,duration,lobe,cog,group_label,cog_labels,stim_index,stim_bipolar_label,stim_shaft,stim_contact_pair,stim_intensity,stim_frequency,pre_start,pre_end,post_start,post_end,stim_start,stim_end
0,FCA13-FCA142.0mA1.0Hz1025µsec,306.718,10,R Temporal,NaN,unknown,[],0,NaN,FCA,13-14,2.0,1.0,303.618,306.618,316.818,319.818,306.718,316.718
1,FCA7-FCA82.0mA1.0Hz1025µsec,351.843,10,R Temporal,NaN,unknown,[],1,NaN,FCA,7-8,2.0,1.0,348.743,351.743,361.943,364.943,351.843,361.843
2,FCA2-FCA32.0mA1.0Hz1025µsec,396.000,10,R Temporal,NaN,unknown,[],2,NaN,FCA,2-3,2.0,1.0,392.900,395.900,406.100,409.100,396.000,406.000
3,FCA1-FCA22.0mA1.0Hz1025µsec,466.968,10,R Temporal,NaN,unknown,[],3,NaN,FCA,1-2,2.0,1.0,463.868,466.868,477.068,480.068,466.968,476.968
4,FCA13-FCA142.0mA7.0Hz1025µsec,547.062,10,R Temporal,NaN,unknown,[],4,NaN,FCA,13-14,2.0,7.0,543.962,546.962,557.162,560.162,547.062,557.062
5,FCA7-FCA82.0mA7.0Hz1025µsec,605.812,10,R Temporal,NaN,unknown,[],5,NaN,FCA,7-8,2.0,7.0,602.712,605.712,615.912,618.912,605.812,615.812
6,FCA2-FCA32.0mA10.0Hz1025µsec,712.031,10,R Temporal,NaN,unknown,[],6,NaN,FCA,2-3,2.0,10.0,708.931,711.931,722.131,725.131,712.031,722.031
7,FCA1-FCA22.0mA10.0Hz1025µsec,767.531,10,R Temporal,NaN,unknown,[],7,NaN,FCA,1-2,2.0,10.0,764.431,767.431,777.631,780.631,767.531,777.531
8,CU_10-CU_112.0mA1.0Hz1025µsec,866.312,10,R Occipital,NaN,unknown,[],8,NaN,CU,10-11,2.0,1.0,863.212,866.212,876.412,879.412,866.312,876.312
9,CU_8-CU_92.0mA1.0Hz1025µsec,914.968,10,R Occipital,['negatif'],negatif,[],9,NaN,CU,8-9,2.0,1.0,911.868,914.868,925.068,928.068,914.968,924.968


In [9]:
# répartition des essais par groupe principal
stims_df = exports["stims_df"]
cond_index_main = build_main_condition_index(
    stims_df=stims_df,
    min_trials=stats_cfg.min_trials_per_condition)

print('catégories principales :', {k: len(v) for k, v in cond_index_main.items()})

# répartition des essais avec sous-catégories cognitives
cond_index_sub = build_cog_subcategory_index(
    stims_df=stims_df,
    min_trials=stats_cfg.min_trials_per_condition,
    keep_main_groups=stats_cfg.keep_main_groups_in_subgroup_mode,
)

print('avec sous-catégories cognitives :', {k: len(v) for k, v in cond_index_sub.items()})

catégories principales : {'negatif': 5}
avec sous-catégories cognitives : {'negatif': 5}


In [11]:
# test manuel pooled sur conditions local et negatif :
input_root = Path(stats_cfg.input_root)
session_dirs = sorted([p for p in input_root.iterdir() if p.is_dir()])

X_all, obs_df = stack_condition_locality_maps_across_sessions(
    session_dirs=session_dirs,
    condition_name="negatif",
    locality="local",
    cfg=stats_cfg,
    subgroup_mode=False,
)

print("X_all shape :", X_all.shape)
print("n sessions :", obs_df["session"].nunique())
print("n unique trials :", obs_df[["session", "trial_idx"]].drop_duplicates().shape[0])
print("n unique channels :", obs_df[["session", "channel_name"]].drop_duplicates().shape[0])

obs_df

X_all shape : (38, 20, 384)
n sessions : 1
n unique trials : 5
n unique channels : 23


,session,condition,trial_idx,channel_name,channel_shaft,stim_shaft,label_stim,group_label,cog_labels,stim_bipolar_label,stim_contact_pair,locality,metric
0,P101_DC54_stim2,negatif,31,C1-C2,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
1,P101_DC54_stim2,negatif,31,C2-C3,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
2,P101_DC54_stim2,negatif,31,C3-C4,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
3,P101_DC54_stim2,negatif,31,C4-C5,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
4,P101_DC54_stim2,negatif,31,C5-C6,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
5,P101_DC54_stim2,negatif,31,C6-C7,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
6,P101_DC54_stim2,negatif,31,C7-C8,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
7,P101_DC54_stim2,negatif,31,C11-C12,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],NaN,1-2,local,logratio
8,P101_DC54_stim2,negatif,35,OT1-OT2,OT,OT,OT_6-OT_72.0mA1.0Hz1025µsec,negatif,[],NaN,6-7,local,logratio
9,P101_DC54_stim2,negatif,36,OT1-OT2,OT,OT,OT_4-OT_52.0mA7.0Hz1025µsec,negatif,[],NaN,4-5,local,logratio


In [12]:
# stats sur une session
out_session_stats = run_session_condition_stats(
    session_dir=session_dir,
    cfg=stats_cfg,
)

out_session_stats


=== Session P101_DC54_stim2 ===
[INFO] P101_DC54_stim2: condition main 'negatif' | n_trials=5


/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] P101_DC54_stim2: condition sub 'negatif' | n_trials=5
[OK] P101_DC54_stim2: résumé sauvé -> /home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_stats_conditions/P101_DC54_stim2/summary_stats.csv


PosixPath('/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_stats_conditions/P101_DC54_stim2')

In [ ]:
# résumé des stats de la session
summary_stats_file = out_session_stats / "summary_stats.csv"
summary_stats_df = pd.read_csv(summary_stats_file)

# d'abord les résultats avec le plus de bins significatifs :
summary_stats_df.sort_values("n_sig_bins", ascending=False)

#### Stats sur toutes les sessions

In [ ]:
pour ces analyses, je m'inspire de "Category-specific visual responses: an intracranial study
comparing gamma, beta, alpha, and ERP response selectivity" par Vidal (2010), et ils font d'abord Morlet de manière exploratoire, puis 

In [10]:
summary_all_stats = run_all_sessions_stats(stats_cfg)
summary_all_stats

/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] résumé pooled sauvé -> /home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_stats_conditions/pooled_across_sessions/summary_stats_pooled.csv


{'config': {'root_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog',
  'input_root': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_morlet_exploratoire',
  'output_root': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_stats_conditions',
  'metric': 'logratio',
  'run_wilcoxon_fdr': True,
  'run_cluster_perm': True,
  'alpha_fdr': 0.05,
  'cluster_alpha': 0.05,
  'n_permutations': 2000,
  'cluster_threshold_p': 0.05,
  'tail': 0,
  'seed': 13,
  'min_trials_per_condition': 5,
  'make_main_groups': True,
  'make_cog_subgroups': True,
  'keep_main_groups_in_subgroup_mode': True,
  'localities_to_test': ('local', 'distant'),
  'group_across_sessions': True,
  'also_run_per_session': False,
  'pooled_output_subdir': 'pooled_across_sessions',
  'fallback_compute_metric_from_raw': True,
  'baseline_stat_fallback': 'median',
  'eps': 1e-12,
  'save_figures': True,
  'figure_dpi': 150,
  'cmap': 'R

In [4]:
# pour avoir seulement les lineplots récap par bande de fréquence
generate_pooled_band_lineplots(stats_cfg)

In [ ]:
# en groupant avec conditions inter-sessions : s
pooled_out = run_pooled_condition_stats(stats_cfg)
pooled_out

In [ ]:
# résumé d'exécution
run_summary_file = Path(stats_cfg.output_root) / "run_summary_stats.json"
pd.read_json(run_summary_file)

In [ ]:
# Variante avec seulement cluster_perm
stats_cfg_cluster_only = StatsConfig(
    **{
        **stats_cfg.__dict__,
        "run_wilcoxon_fdr": False,
        "run_cluster_perm": True,
    }
)

In [ ]:
# Variante avec comparaisons sur zscore plutot que logratio
stats_cfg_subtract = StatsConfig(
    **{
        **stats_cfg.__dict__,
        "metric": "zscore",
    }
)

In [ ]:
# Variante sans subtypes cognitifs
stats_cfg_main_only = StatsConfig(
    **{
        **stats_cfg.__dict__,
        "make_cog_subgroups": False,
    }
)